# Lab 17-08 · Block, Convolutional, LDPC and Polar Codes

Six parts, each built on `coding.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
from scipy.stats import norm
import coding as cd
rng = np.random.default_rng(1708)

## Part 1 · Hamming codes

*Predict:* the number of weight-3 codewords of the (15,11) Hamming code, and its block error rate on a BSC with $p = 10^{-2}$.

In [2]:
G, H = cd.hamming(4)
wd = cd.weight_distribution(G)
print("(15,11) weight distribution:", dict((w, int(a)) for w, a in enumerate(wd) if a))
print("G H^T = 0:", not cd.gf2_matmul(G, H.T).any(), " d_min =", cd.min_distance(G))
p, n_blocks = 1e-2, 50_000
msgs = rng.integers(0, 2, (n_blocks, 11)).astype(np.uint8)
cw = cd.gf2_matmul(msgs, G).astype(np.uint8)
rx = cw ^ (rng.random(cw.shape) < p).astype(np.uint8)
dec = np.array([cd.syndrome_decode(r, H) for r in rx])
sim = np.mean((dec != cw).any(axis=1))
print(f"block error: simulated {sim:.4f}, formula {cd.block_error_bsc(15, 1, p):.4f}")
assert wd[3] == 35 and abs(sim / cd.block_error_bsc(15, 1, p) - 1) < 0.1

(15,11) weight distribution: {0: 1, 3: 35, 4: 105, 5: 168, 6: 280, 7: 435, 8: 435, 9: 280, 10: 168, 11: 105, 12: 35, 15: 1}
G H^T = 0: True  d_min = 3


block error: simulated 0.0100, formula 0.0096


## Part 2 · CRC bursts

*Predict:* which burst lengths CRC6 ($x^6 + x^5 + 1$) always detects, and the fraction of 7-, 8- and 9-bit bursts it misses.

In [3]:
g6 = cd.poly_bits(cd.NR_CRC["CRC6"])
for L in range(2, 10):
    miss = 0
    for mid in range(2 ** (L - 2)):
        b = [1] + [(mid >> (L - 3 - i)) & 1 for i in range(L - 2)] + [1]
        word = np.zeros(L + 10, np.uint8); word[3:3 + L] = b
        miss += cd.crc_check(word, g6)
    print(f"burst length {L}: {miss} of {2 ** (L - 2)} patterns undetected ({miss / 2 ** (L - 2):.4f})")

burst length 2: 0 of 1 patterns undetected (0.0000)
burst length 3: 0 of 2 patterns undetected (0.0000)
burst length 4: 0 of 4 patterns undetected (0.0000)
burst length 5: 0 of 8 patterns undetected (0.0000)
burst length 6: 0 of 16 patterns undetected (0.0000)
burst length 7: 1 of 32 patterns undetected (0.0312)
burst length 8: 1 of 64 patterns undetected (0.0156)
burst length 9: 2 of 128 patterns undetected (0.0156)


## Part 3 · Viterbi decoding

*Predict:* the union-bound BER of the $K$ = 3 and $K$ = 7 codes at 3.5 dB, and how much more $E_b/N_0$ hard decisions need.

In [4]:
codes = {"K=3 (7,5)": (("7", "5"), 3), "K=7 (171,133)": (("171", "133"), 7)}
for name, (g, K) in codes.items():
    spec = cd.distance_spectrum(g, K, 50)
    nbits = 100_000 if K == 3 else 600_000   # errors come in bursts, so the K = 7 estimate needs a long run
    u = rng.integers(0, 2, nbits).astype(np.uint8)
    y, _ = cd.bpsk_awgn(cd.conv_encode(u, g, K), 3.5, 0.5, rng)
    sim = np.mean(cd.viterbi(y, g, K) != u)
    print(f"{name}: d_free = {min(spec)}, union bound {cd.union_bound_ber(spec, 0.5, 10 ** 0.35):.2e}, simulated {sim:.2e}")
g, K = codes["K=7 (171,133)"]
for eb in (3.5, 4.5, 5.5):
    u = rng.integers(0, 2, 100_000).astype(np.uint8)
    y, _ = cd.bpsk_awgn(cd.conv_encode(u, g, K), eb, 0.5, rng)
    print(f"K=7 hard at {eb} dB: BER {np.mean(cd.viterbi((y < 0).astype(np.uint8), g, K, soft=False) != u):.2e}")

K=3 (7,5): d_free = 5, union bound 2.75e-03, simulated 1.56e-03


K=7 (171,133): d_free = 10, union bound 1.15e-04, simulated 5.00e-05


K=7 hard at 3.5 dB: BER 1.45e-02


K=7 hard at 4.5 dB: BER 1.97e-03


K=7 hard at 5.5 dB: BER 1.70e-04


## Part 4 · LDPC

*Predict:* the $E_b/N_0$ at which a rate-1/2 LDPC code of length 504 starts to work, and how the iteration count changes above it.

In [5]:
H = cd.gallager_ldpc(504, 3, 6, np.random.default_rng(3))
Gl = cd.gf2_nullspace(H)
print(f"H is {H.shape}, rank {cd.gf2_rank(H)}, so k = {Gl.shape[0]} and R = {Gl.shape[0] / 504:.3f}")
for eb in (1.5, 2.0, 2.5, 3.0):
    fe, its = 0, []
    for _ in range(100):
        c = cd.encode_block(rng.integers(0, 2, Gl.shape[0]), Gl)[0]
        _, llr = cd.bpsk_awgn(c, eb, Gl.shape[0] / 504, rng)
        hard, n_it, ok = cd.min_sum_decode(llr, H)
        fe += not (hard == c).all(); its.append(n_it)
    print(f"{eb} dB: frame errors {fe}/100, mean iterations {np.mean(its):.1f}")

H is (252, 504), rank 250, so k = 254 and R = 0.504


1.5 dB: frame errors 31/100, mean iterations 25.2


2.0 dB: frame errors 8/100, mean iterations 13.1


2.5 dB: frame errors 2/100, mean iterations 7.1


3.0 dB: frame errors 0/100, mean iterations 4.6


## Part 5 · Polar codes

*Predict:* how the fraction of 'undecided' channels (erasure between 0.1 and 0.9) changes from $N$ = 8 to 1024 at ε = 0.5.

In [6]:
for n in (3, 6, 10):
    z = cd.bec_polarize(0.5, n)
    plt.plot(np.linspace(0, 1, len(z)), np.sort(z), label=f"N = {2 ** n}")
    print(f"N = {2 ** n:5d}: undecided fraction {np.mean((z >= 0.1) & (z <= 0.9)):.3f}, mean capacity {np.mean(1 - z):.3f}")
plt.xlabel("channel rank (sorted)"); plt.ylabel("erasure probability"); plt.legend(); plt.show()
frozen = cd.polar_design(256, 128)
for eb in (2.0, 3.0, 4.0):
    fe = 0
    for _ in range(200):
        u = np.zeros(256, np.uint8); u[~frozen] = rng.integers(0, 2, 128)
        _, llr = cd.bpsk_awgn(cd.polar_encode(u), eb, 0.5, rng)
        uh, _ = cd.polar_sc_decode(llr, frozen)
        fe += not (uh == u).all()
    print(f"(256,128) polar, SC decoding, {eb} dB: frame errors {fe}/200")

N =     8: undecided fraction 0.750, mean capacity 0.500
N =    64: undecided fraction 0.312, mean capacity 0.500
N =  1024: undecided fraction 0.131, mean capacity 0.500


(256,128) polar, SC decoding, 2.0 dB: frame errors 27/200


(256,128) polar, SC decoding, 3.0 dB: frame errors 4/200


(256,128) polar, SC decoding, 4.0 dB: frame errors 0/200


## Part 6 · Reed–Solomon

*Predict:* the raw BER at which KP4 and KR4 reach a codeword failure rate of $10^{-12}$.

In [7]:
ber = np.logspace(-6, -3, 200)
for (n, k), name in (((544, 514), "RS(544,514) KP4"), ((528, 514), "RS(528,514) KR4")):
    plt.loglog(ber, [cd.rs_codeword_error(n, k, 10, b) for b in ber], label=name)
    thr = 10 ** optimize.brentq(lambda lb: math.log10(cd.rs_codeword_error(n, k, 10, 10 ** lb)) + 12, -7, -2)
    print(f"{name}: t = {(n - k) // 2}, overhead {100 * (n / k - 1):.2f}%, raw BER for 1e-12: {thr:.2e}")
plt.ylim(1e-30, 1); plt.xlabel("raw bit-error rate"); plt.ylabel("codeword failure"); plt.legend(); plt.show()

RS(544,514) KP4: t = 15, overhead 5.84%, raw BER for 1e-12: 2.43e-04
RS(528,514) KR4: t = 7, overhead 2.72%, raw BER for 1e-12: 2.30e-05
